In [4]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

df = pd.read_csv('train.csv')

Task 1: Identify Data Quality Issues
Checking for missing values and data types to identify inconsistencies.

In [5]:
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])
df.info()

LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   objec

Task 2: Handle Missing Values
Strategy: Imputing missing values using the Median.
Reason: We use the median because it is robust to outliers, which are common in house prices.

In [6]:
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())
print(df['LotFrontage'].isnull().sum())

0


Task 3: Detect and Handle Outliers using IQR
Method: Interquartile Range (IQR). Identifying and removing extreme price values.

In [7]:
Q1 = df['SalePrice'].quantile(0.25)
Q3 = df['SalePrice'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_clean = df[(df['SalePrice'] >= lower_bound) & (df['SalePrice'] <= upper_bound)]
print(len(df), len(df_clean))

1460 1399


Task 4: Normalization & Standardization
Goal: Scaling numerical features to a standard range.

In [8]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

min_max_scaler = MinMaxScaler()
df_clean['SalePrice_MinMax'] = min_max_scaler.fit_transform(df_clean[['SalePrice']])

std_scaler = StandardScaler()
df_clean['SalePrice_ZScore'] = std_scaler.fit_transform(df_clean[['SalePrice']])

print(df_clean[['SalePrice', 'SalePrice_MinMax', 'SalePrice_ZScore']].head())

   SalePrice  SalePrice_MinMax  SalePrice_ZScore
0     208500          0.568994          0.646235
1     181500          0.480498          0.190222
2     223500          0.618158          0.899575
3     140000          0.344477         -0.510685
4     250000          0.705015          1.347142


/var/folders/24/0j56gldd1qqcftrzcsmdlspw0000gn/T/ipykernel_15929/14609073.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['SalePrice_MinMax'] = min_max_scaler.fit_transform(df_clean[['SalePrice']])
/var/folders/24/0j56gldd1qqcftrzcsmdlspw0000gn/T/ipykernel_15929/14609073.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['SalePrice_ZScore'] = std_scaler.fit_transform(df_clean[['SalePrice']])


Task 5: Principal Component Analysis (PCA)

In [9]:
from sklearn.decomposition import PCA

features = df_clean[['GrLivArea', 'GarageArea']]
pca = PCA(n_components=1)
df_clean['Area_Component'] = pca.fit_transform(features)

print(pca.explained_variance_ratio_)

[0.87831312]


/var/folders/24/0j56gldd1qqcftrzcsmdlspw0000gn/T/ipykernel_15929/1079384021.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Area_Component'] = pca.fit_transform(features)
